# Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import base64
import io
import re
import zipfile
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Our submission declares gemma-4-31b-it-qat-w4a16-ct in agent.yaml; no rewrite needed.
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'

# EXP-002 shim check: unpack OUR submission (dist/submission.zip, sha256
# 33eed6f5d49e2b73f6b488f637ea5c86f2d5d1da5cbf42bd7f819ddb3cd9098c) instead of sample_submission.
AGENT_DIR = WORKING_DIR / 'our_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVwJEt0MXAIAAAgEAAAKAAAAYWdlbnQueWFtbF1TTY/USAy9t9T/4cFceqRJtwYGDs0JaXuBw66gZ4XEKXKnnMSa+qLsTOhftP9jfxlKwiwNpypbLvu951dXOKZkoI6joU0FOpyCqEqKeLzFJpcUslUp+vMNVGLneSm+QUwgR9m4zHd9EO/1erteXeG+p8xok/dpVFjPeP/2+Pfh/r4+Ht7+8dcBQUpJBRQd2jQUNCmEIYqdL+YrNtlT3AaH//59ub3DpxdL98+H44c/v2w+vbjew0nbwnpRtDJDI4lqcGS0KzTulEL2XP/supvRb88UPFJseEY3lSsbROHSGH0ix+6XWbfXe1hKHpECK07s04gwqIFyZiqQOHdqUsjiuTIJjH9S8kfuRK2csekKZ5B7qC4kHntmP7Ga2u6hI9dNclzWq5Ac+z06DoGqu+rl7akSq76SVeMd3b6uGgPwqxYLVwQ6QzN7DzGMYj0IuaRHcVyQC7fy7c2PkxVUGGpFcmaH03km0VOJrLpeOdamSDZJcY8ja/KPrEiR8U7s/XCCqA78xP3j2foUUTgnFUvlDDLsxlQeNFPDT1iCRAnkb/DIRVphh0zW9Nv1alpdGZpl3DOJjR8cT9hDNt39L842uPWq48iFbEpF42jT2Up38W5J6GIBid288/Vq2qLu1yugQhliPVmPovuRYHL15KQlZCd2EY5FjC/ijq1WIxt0iefFWj3TmTJXeFco97NxFqUdN57KpPRgcBySsXuSb+GJzfPps0FazIZpSbw+v36DwyvQyZPx/KECUttuf8KYhKkjS9efUnmCw1SavlYJ4qnMJb890OHUTQjXq+9QSwMEFAAAAAgAAAAhXN159oETAgAAkAMAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWx9ks9u2zAMxu8B/A4fkEuCGc4/N2td9JBhHjas7bB2GLCTodi0I0SWNFFO6ifqe+zJBjlDMLTYdCFEUfz4IzlGQ5qc8FSURnvSPthaNqiNAx3I9RANaY+JVUInbYVfz2myAnvhvNQNDkJ1xNMkGo2xUcocqUItSVUMSw4fNw/3+eNj8ZBv3t/laKVzxmXw1Nog2zmK4Y0t7MnsY7TiqTCdt50vvNmT5hjWEZMuqbCkhfJ9HMRqRz870mV/9oJDCj65iWM4Yms0U9HKlgrfW4rBRFUMv5N6L3XzhzaJRn9VlGGeLHE+Y3BpHDlUVItOeUjGIplfI0+xmb3jIfow+KLRwBISXF2cLvsM6zQavaLKkM6v1kP6PZFlWGda6/EGpzAcSSl0uiIHvyOslvHb9SWGKT15HIVS18gXC3gnibGcp5ez9SJNo9ELtiwaAVKXqquo8DvTNTvPGbzrKIjXQjHh5twRmLrGhMPIBYO02Krw7fR4M0RPr5EvZ/kKYquEp/CVg8hZeNtVDfkMy4sAiDG+5w+fPvyYfF1NM5SmbTstfQ82nSsJjkQVhFn0PKDuhNPEjMoQ4/7Lt7CLR+Gqs9C/zhgnZXiDw+3tHSZGq/4lBIyembqexmDzetvCdEMRjoRCKWzyf8HPRBbSQ+j+KPos1N6qULysIRttXNi2rdQDoawhznSdrULzSNfGlcSQPolGvwFQSwMEFAAAAAgAAAAhXPryLAxGAgAAKgQAABAAAABldmFsX2NvbmZpZy55YW1sfZNNbhQxEIX3I80dnpQFMyLp+SEBMYEFC5CQEAk/ArHqVNo13da07ZarzKRPxD04GbInRAFBetWyy/W991w+wiXHEyXZ4TqZllUwG3rylTP4+eO0Op1X+NwxpAmRIyKTEfANNdqP0I6FsQ0pYsejIHnDEVf8nfpEaoPfXJ1PJ0cgOCtifZvL4Ji84P0FeuusHoO8yZ3QUfQsAsNbSr1i9nQJZ/0ii5tjH1JvcN2HfSlerdEhJn+rusqYL68/vn3zbfZhPd+gCX5roytAT47lDvOHPHgWzcKoJetFYUhpEWm/EHJDz7Wk66I9+EU+V5e+bTWS6wvza4iiaEgYA0fk6DZYrZfIqgU3mJ1WZ3iMVVXMQFjTMMdLrPJKhxd3fy0pY6Yh9LLIohwp17fuhnFeTSf3hE8ngKObWq3j2lmflGWDjAKOcpIGDQ0Ivlimlr2iD2HAjGT0jQ3IJ0PS+TnYWJVcaqzsQJEhavsebSTD5o4UQl831PeZkzGF1LLWoqTpkG9JS+uBtOlKp21kLjsmwAdFE5LXu5Ypetng2TIv3OqphZvgjWzwZLk8MO5fa864Cc4lb3WEhBQbLkOZ71A7K3j17tNF9i7F+WVHwifr4iWXDKOyFAH/+47yVB1DAvaMHfPwz9m0XpTJIGzLlT8SrM6WkMNbKXGfiDX8MCgbKVN5sA4reURmfyVxjMiOrC8O8zvN+/OiMEfUBFHJ2XbWt+cPA7OTIQY3KPYxR3SIA9bj6reI58sraMB1SN7AKsiPexqr6eQXUEsDBBQAAAAIAAAAIVxjR9J2AxAAADkoAAAUAAAAcHJvbXB0cy9zd2VfY29kZXIubWTdWt1yG7mVvudTnEouRE7ItuzxTLJUKVWesZxxJWO5bE+y3lSKArtPsxGhgTaAJsWtvdiH2CfcJ9n6DtBkU9LsaG6jK6rxd3D+z3fw2fWkPFPY8ap0Ffs5KQpstfP0fh8bZ4ntRltmT7W+03ZD1++uSIfQM2lLsWHy3Lmgo/N7UpGe7Zy/DZ0quaAfdFWxpcghBqqdp9jokBdvvKqY9q731KlYNkvSkToVAgdy1uxJ1xQbFzgvxxApWxHf6RBBSPp+y9zJoLabgj67nkAAubo22jL1tmJPihrlK9opYxalceWt7BSdM4tSGUPrvtpwLOhvWFqrEOcUotoTb3XFtuTFWgWu5rIq9OtWR1JknV1w28V9ugCtuXaehSVpP2JbhWIy+S19OpxU6VDqTkib8pb9nvB5TtYR35XcRe1smE0WdHWnymj2wm4QKvOoY0+x97ag79JpOs5p53Vk8L51IVLcOQqN8xFyjKA+XJDFWeQ5RBUTiUkKzhMrbzR7cn3s+ljQn8FQZffUJOl5VsFZYbgjRTXvjhvPsZUlVcZisqBPDSeeWtUyBeMiNc5Uga7f/eUzOcvkIFMOTJgRluR7uypd2ypbzXFQtaq14TlxpWP+KZfLvzccV7hBH+ZZDCthfRqB/q4s602zdh4zWPmyWQXdaqO8DI8mhn698aprCnonvOl6yDQ0bAwdSFKEgyHfBqwCV1xsIAO+i4P+H24LHrzym75lG4OYFYagsSLKk+tSVLcchpOK4+XzAH7KsZ3pAznRC2WglT6uRH3EFmyV/plqG3nDPsyKI/PubTUnZ6pViF7bzZws7/LvYsTjB0tKB1HHYsT7kRUk9ssiaLAaLl88IpC8tXUVF4/JJo9/6dnvi4dyGi0PNFVkdIizQXjabtnGLJuBiqRkQm1S/2Qnt7y/3Coj/ivoikmJaUF4b2tS1HneatcHWEvnbGDaqUBlH+FTYGWQeXS3bMnoFvZXObIuwg9yHj7YzJIYzkLUBCpztGTdtlxpFdnsE0M7oyMZ5TcsEgykbXQUWmUMrNNyciXfJddiuNqwn3yEPlByJPALtNNQVcu09qq8xURteS7Ol+9U2xle0t+7RgW+vHr99pPQEi6f//7Zy2/SsZfPs2wjV5fW/aOg711vY/LUbmfTiouxPuhAteekkRu95ZA0nmzfrhmW2Ac4KlJr1w/EPn+RNpJFu4aThHobes8FXcmcb9KUOSkT3ODl6OuFaPzHT68+XdG7609XsMPOKFgjWDyF8gaKri8b+Oyy9x7a0Ow76EfQAcp/FylE7mYXB/8noklOMBlv5V3XcUW1d226vhjDXUxxQGKZdRGulHaNihR6v8X1RU4/IOD43nCYLOhj6cVQEmkbB1f6LLadRLrk4yJn4Y2dxFRRw54rV1Loy4ZUWFKpIv1RVj/z3HlXdHuqnTFuxxWt96JrofS6i0KmsnR1/YZa5W/Zzwp6nZQVIhmZPfQjjIlcjkeV2al9SF9CDqnHOJ/Yoew+NogShmtxjccJtObSwRQ7KKurR1G/gM3tXS9aQaVnFbmCHx7RcrrZnCo2DK7Hh/zKITgrsPi2qyGqLknRlm3lPFfUqfJWbTjJ9jd/tlDsoGy1dne0UV34Da3ZuB1EC27dI4i3yGoa128akGFE5++zhf73v/8nO54DxXAyyWO1rtL1HkElXTplM+Jva/wsuv2cur381Fbjd+fdP7mMRXStQXSLfVeU9QZ7RHeHWUsRvmRWEuw5BnxpC3qj72TM6LVXSDrghSeLQRu0DVHSi8SWZJXZrUW4FgeXBk+PzMFG9pbjRVK1zDUdDjmXRCZYcFIIy1xxhQnKIMzth+O4OhGPrukm3ZgWX27I2aQnES4xJsbXShuYdpJA6YzhMqvBzY+u6g2/c/GN62115b3zS3rnwOneDKH4TFuQuApWdaFx8ewG7Ls5q7SP+xV/6ZUJZzdIa1Qk3XZIoxhbgfzeejaioNEl2mp9Bxu9Ff3ZqC6lBDoMXDkLIteFJLQVd2yRTmpGmP7I/P9qng2RVQVz2egt+NjDXaToo8LtUZV8n77uGocADv6FXkcu6E9eVVjp2fc2DGECE7QlkadwNfm2RtkNZ7cpEVb2W8iMQS7ICZ5F9wz/rO6Kbn8jmSuCMgWGMLSzszkFR1v2UHAJiPCNSJI2XtneKK/jfpnSJtCdcrtsZtpuICxsqHPC3XkGnzMJIuyoW0Ys+bfzgl4N2f+w6hjXU0Yjx0dtTFYft2UPZQ+l8wgUosQtk4KtQ6Rg7XvvInQLo9nt007byu1O07gUMKjtQ6Q10xq6B71+rzvOWY6c4cVZNJDowtLLc6hd+q+kl+fn5xfUYYVcUYQ1LIlKGyz59nxIdo6pYpZRzhrwaTn4aimXfi5bLMECcGyoGZ6/OJdEAYEPN0hM2bdrZ8Q7w4Qlu7VRL0Kn7TJ7tlRFpAyKUCtFXSojlQJy5ezqjQqRvk7RXDy+GhLfwaJ3Gp590MGGSXWdd9DQaaXrmiWCS2o4p+OHpDnH/1uOjatm8Co6io+j58nFbzx31LKyIuM2FZyj44ath0wOTi+5aymHkpreUautbpVJmQ5b6NESbsKwJJyS+sHZHAusihGL1yiVsoZ3XGqFJDCksItUL6dmBf3UVQgGQa8N5HPYWkldONoXIxp2OsrssPvelkPYV/LPVnmtLFLlH1Ouni3uDOqxx5nv4NxrVUapmqyjitf9hjqvLSKSdSKsVNNIBnd0QZPf0kMPRtMUZg9cgw4ENvU8K03NIARXz2FHx9nk087BdfpqgSxhP/Kbx6CE0O5ahvUHanWyepHv2OmehOdTgOIsSAqbkIPs3lOAhFLCnDPyAKUPzKdFa8pqHws0N8gAjxEJZDlxzFI3J881P0T6hg/ED3nIwCI4IYXajKLrFoa3DMjAM4Szz7/M/mGuMVUHnUnkjmCSw0b5sEEMGRqxe7oRhjy7OR41S352XHxLpS+RZMhL6FOCc0Szy4YVzhMcB7Eeph2bujdiNSr2fmAlPJgZUpGzQF2/NrqkV+/fHowk9uvkuEfujEMGJ3BATg5qjar4CCCJH07Ofs2IHIAqkmwgUnFTp8iBOCFkvKvkDhA+ICgRz3ovGpDMFLfiu47LiLIiEdrqIMVvdGCnK+g1Uj0x3fgwGyXPi+j1BoV6FhI06hGFEmr/icvU+g4Z8dFV5gxPtuttMZm8renml/Ock4T+5t7ws9VKWx1XK4T0JPoBtJjc3NxMSoNw8jHPXk6IiFYrYB5htaJLmp6tpKQ+m89krOKahi2nyfRl/PKdszxL6/GHoSItpcs0ZbReTl1tOOrI7Wo1LVEL4vdoB89S9pYmjBbyl8OxAgk8nH9y8GWaNdrA8i9tAPGnTdJpadpoC9RmeZOHqzE4HdEwXtio0PzsQgyeLpxg1SDH6QMu53UfTybMRKiPK85pJnyqNeOxp6nM6jWWfKcCL0dXdP6X2Lu69icTxox92moZQplk6idz162lwDqdNRmucu2nx+vkxY8ouprT+r6KK7okdfppTZe0/vU6qw7qiggz7PRQg5+ofmpGv6Mz+i86o9+NPq8Pt34bXusyji6OfKuMP3/9r5TfhDl99dXtDr9Gh2Nd8Quzn8YQXaeqNUgCYUsepD0i7d6V3ygTkm8ZfcT0bLngofx7IvO34b3yUSvzr8iE1ZCEXGbFn6aTRwuUMWnXYsNxejs/rBF+bSUw3s5pi3gsigPvHKazEQOvLV/XTzKcr8Q3jZklW6avQ3j4dW5e2X2i/0jukdZ83FON5my4y6mp5F1gRrOz4do/qPABM55y7SGlkNPvX/5kkC5PJ/8qVgjFR11/uPuT+TBc7pQPp9fI7EAgyMllH7UJEisGjARpOdWq1Ui5JRcPDxKTYghcN9Tb0tlKp0YIUNPghkw91xsn8elGpK3ufUwVBTKp3nMGlofV946+OSIDOyVI3VA60Q755RFSQPaXM/+CXh16RIN55fgqObFkvp3nxSE9Z7vV3lmpH6W0S6nlSeERd2451Ka51aGOEKYkp7KKx6nxCCBH9ti6LcoZqdkAbrjSGZpKJyDBfZIbl6oLF8PclDejuO9AvSCAXM0mzwv6y/X32H16ABDOU3UvtXf0yGpzD0B6MLlGy8XrkureClQEwBMGg/K4U161HKVhcGiFUtx3GBQMbtFyCAm1VZvcZ/ooEMtDJLgX1m90TKX/IoPrUnuBYCnDa11SZN8KJDT0u6UYmA9ITlp2xG1Q9KSm6NCuA0CkbKWlcJfZiqLeNKn1Qh5FhUBx6GE5ZwJNn9itHHbLQAxY+UgDcyZ18dAhXAqyj5r22EXfucQEgBzADGqBeKwTeFZK36GNWaEIS3UHMOuNimBF6i7kmxojuplLU7ezCQYGYiAQMPoYmJC63cwd+2LyoqAPV+8/XL/+6fuR0nx70Jm/SW/ntKex1QoGM5Lp0AqRwjDBE9KVDYHhO4QMbhS6d/5xCCbVsL19rHmwpLIal9QCPSaVOCFsLkhd9QCgg268rbFzqSwsdS3Ig3dVX2Y9yg3jUnVz6rwrOUHJ0ohTdg83I1SJ/a4BBR6bVsXk6yLNPBpd7qEBTD709Y8r4AkgiaFbT9ORoLSlf0dfRkFdPl8kLAwm8x9ScQZgMgX9BNTj0E5OvcVDD5lK1+lje/to5AkveIGrvaTe6i89Gg8WAkgPE6S7IP4PHvO955qRywoumzqeo1NTm1AsVBRPsE5nWbh9nIZokrRAxTGRaN5CGknpE5AvDTCha/TqIK0UcyW1QTsRZCYocDBEUI8Uq0VdP9xAb6EsZeP88Z2FNE0OwLIMFvSqjoz3FmUjdJ+ezRmOrDIJDhKTwGJr7dti8rKgv159ePvm86Ne94MAMxnt8m5wU709VfEjcD7o9qIdkPXFl6G/MOokgLppmKXz7rc/MoA6fQy7X45RimScdY+AKcC1QG1oUksfXU6CAcDCoZo6jtAOQI2zn7M6wOICKCdqsQgdwhINgINCZpB+17AE5jHtYgAV6XghUGUyT9hEEj4Qliy3F7I7+sH6jsp9KVEiRIfHMlXu6X786bsf3346hhqx4S37gFCGTKVRWy4m3xR5ZkHfQ9nH71rERZ1CVCdPBYQWaeUBVJcuixhhpWNBH49Q07gpXyqbvZH0Vy9SaDhsIqcFobFErx8dd0DIBGkfut5igjUCwc71piKDt1lo2OfHAvLuSrKy/EIKjfX0tiGbk5Be916EoNvOu62A22FgcXIe8s7lhAGeO3Mwwrhz6QVA6NsWvUzJbvILlGH3wzuL1Ix/4zx88PGikw+8Ub4yHBCzSNKgecp3Rg8bQuN2AS+ehDZl6Q+jjckByT4MfXtOgZGhIlFqlQbyO8ecA/gI2SM6VxlG/8M5nnOV8nAlheDRW7SsWnzXGQc/tgRjdEhqBWHLlhL57VzsXIJuw+Vtkt4DqDN1yqSaFRXO7S9xZXID6xKQeoHmwfhhW8Zg84L/ZO+Kyf8BUEsBAhQAFAAAAAgAAAAhXAkS3QxcAgAACAQAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAACFc3Xn2gRMCAACQAwAAFQAAAAAAAAAAAAAApAGEAgAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAAAhXPryLAxGAgAAKgQAABAAAAAAAAAAAAAAAKQBygQAAGV2YWxfY29uZmlnLnlhbWxQSwECFAAUAAAACAAAACFcY0fSdgMQAAA5KAAAFAAAAAAAAAAAAAAApAE+BwAAcHJvbXB0cy9zd2VfY29kZXIubWRQSwUGAAAAAAQABAD7AAAAcxcAAAAA"
zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Phase 1 Inference and Phase 2 Verification

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read optional evaluation settings from the submission's eval_config.yaml / eval_config.yml
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# EXP-004 shim fix check: exp003 showed fastapi_14361's agent correctly vendors
# inline_snapshot, verifies tests pass, then runs `rm -rf inline_snapshot` and resubmits
# without it (experiments/README.md 1l) - swe_coder.md's scratch-file cleanup rule
# conflicted with the Known-sandbox-gaps instructions. Patched swe_coder.md with an
# explicit carve-out + a keep-it-in-the-patch instruction. Re-running only fastapi_14361
# (the one task that reliably reaches VERIFY within budget) to confirm the shim now
# survives into the final submit_patch call.
TARGET_IDS = {'fastapi_14361'}
SAMPLE_TASKS = [t for t in tasks if t.instance_id in TARGET_IDS]
assert len(SAMPLE_TASKS) == len(TARGET_IDS), f'missing tasks: {TARGET_IDS - {t.instance_id for t in SAMPLE_TASKS}}'
limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    patch = (result.agent_patch or '').strip()
    predictions.append({'id': task.instance_id, 'prediction': patch if patch else 'NO_PATCH'})
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(patch)}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 6. Package Submission Archive

In [ ]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Validate unpacked submission directory structure, file types, symlinks, and size limits
for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')